In [5]:
import os
import json
import re
import unicodedata
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Função para limpar a pasta "./models/"
def clear_models_folder(folder_path='./models/'):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.remove(file_path)
                elif os.path.isdir(file_path):
                    # Se houver subpastas, removê-las recursivamente
                    import shutil
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Não foi possível remover {file_path}. Motivo: {e}')
    else:
        os.makedirs(folder_path)

# Limpar a pasta "./models/" antes de iniciar
clear_models_folder('./models/')

# Função para limpar e normalizar o texto:
def clean_text(text):
    # Remove acentos (normalização Unicode)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8', 'ignore')
    # Converte para minúsculas
    text = text.lower()
    # Remove caracteres que não sejam letras, números ou espaços
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Substitui múltiplos espaços por um único espaço e remove espaços nas extremidades
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Carregar o dataset (arquivo JSON com dados sintéticos e reais)
with open('./data/synthetic_and_real_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

labels_text = []      # Para armazenar os rótulos (labels) em forma de string
descriptions = []     # Para armazenar as descrições

for d in data:
    # Processar a descrição:
    # Se existir o campo "description", usá-lo; caso contrário, usa "trecho"
    desc = d.get("description", d.get("trecho", ""))
    # Remover os tokens especiais temporariamente para limpeza
    desc = desc.replace("startseq", "").replace("endseq", "").strip()
    # Limpar e normalizar o texto
    desc_clean = clean_text(desc)
    # Re-adicionar os tokens especiais de início e fim
    desc_clean = "startseq " + desc_clean + " endseq"
    descriptions.append(desc_clean)
    
    # Processar os rótulos:
    # Se existir o campo "labels", usá-lo; caso contrário, "temas"
    labs = d.get("labels", d.get("temas", []))
    # Junta os labels em uma única string e limpa
    labs_text = clean_text(" ".join(labs))
    labels_text.append(labs_text)

# Criar os tokenizadores com um token para palavras fora do vocabulário (<UNK>)
label_tokenizer = Tokenizer(oov_token="<UNK>")
label_tokenizer.fit_on_texts(labels_text)

# Para as descrições, usamos filters='' para não remover nenhum caractere e preservar os tokens especiais
description_tokenizer = Tokenizer(oov_token="<UNK>", filters='')
description_tokenizer.fit_on_texts(descriptions)

# Converter os textos em sequências numéricas
label_seq = label_tokenizer.texts_to_sequences(labels_text)
desc_seq = description_tokenizer.texts_to_sequences(descriptions)

# Definir comprimentos máximos:
# Limitamos o tamanho para evitar sequências muito longas (ex.: 20 para labels, 50 para descrições)
max_label_length = min(max(len(seq) for seq in label_seq), 20)
max_desc_length = min(max(len(seq) for seq in desc_seq), 50)

# Aplicar padding para padronizar o tamanho das sequências
label_padded = pad_sequences(label_seq, maxlen=max_label_length, padding='post')
desc_seq_padded = pad_sequences(desc_seq, maxlen=max_desc_length, padding='post')

# Dividir os dados em conjuntos de treino e teste (80% treino e 20% teste)
label_train, label_test, desc_train, desc_test = train_test_split(
    label_padded, desc_seq_padded, test_size=0.2, random_state=42
)

# Salvar os dados pré-processados (em formato .npy)
np.save('./models/label_train.npy', label_train)
np.save('./models/label_test.npy', label_test)
np.save('./models/desc_train.npy', desc_train)
np.save('./models/desc_test.npy', desc_test)

# Salvar os tokenizadores em arquivos .pkl para uso futuro
with open('./models/label_tokenizer.pkl', 'wb') as handle:
    pickle.dump(label_tokenizer, handle)
with open('./models/description_tokenizer.pkl', 'wb') as handle:
    pickle.dump(description_tokenizer, handle)

# Salvar os parâmetros de pré-processamento (comprimentos máximos)
preprocess_params = {'max_label_length': max_label_length, 'max_desc_length': max_desc_length}
with open('./models/preprocess_params.pkl', 'wb') as handle:
    pickle.dump(preprocess_params, handle)

print("Pré-processamento aprimorado concluído e dados salvos.")
# 6.9s

Pré-processamento aprimorado concluído e dados salvos.


In [6]:

import os
import json
import re
import unicodedata
import pickle
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Verifica se há GPU disponível e configura o crescimento de memória
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU(s) detectada(s) e configurada(s).")
    except RuntimeError as e:
        print("Erro ao configurar GPU: ", e)
else:
    print("Nenhuma GPU detectada, usando CPU.")

# Função para limpar a pasta "./models/"
def clear_models_folder(folder_path='./models/'):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.remove(file_path)
                elif os.path.isdir(file_path):
                    # Se houver subpastas, removê-las recursivamente
                    import shutil
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Não foi possível remover {file_path}. Motivo: {e}')
    else:
        os.makedirs(folder_path)

# Limpar a pasta "./models/" antes de iniciar
clear_models_folder('./models/')

# Função para limpar e normalizar o texto:
def clean_text(text):
    # Remove acentos (normalização Unicode)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8', 'ignore')
    # Converte para minúsculas
    text = text.lower()
    # Remove caracteres que não sejam letras, números ou espaços
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Substitui múltiplos espaços por um único espaço e remove espaços nas extremidades
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Carregar o dataset (arquivo JSON com dados sintéticos e reais)
with open('./data/synthetic_and_real_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

labels_text = []      # Para armazenar os rótulos (labels) em forma de string
descriptions = []     # Para armazenar as descrições

for d in data:
    # Processar a descrição:
    # Se existir o campo "description", usá-lo; caso contrário, usa "trecho"
    desc = d.get("description", d.get("trecho", ""))
    # Remover os tokens especiais temporariamente para limpeza
    desc = desc.replace("startseq", "").replace("endseq", "").strip()
    # Limpar e normalizar o texto
    desc_clean = clean_text(desc)
    # Re-adicionar os tokens especiais de início e fim
    desc_clean = "startseq " + desc_clean + " endseq"
    descriptions.append(desc_clean)
    
    # Processar os rótulos:
    # Se existir o campo "labels", usá-lo; caso contrário, "temas"
    labs = d.get("labels", d.get("temas", []))
    # Junta os labels em uma única string e limpa
    labs_text = clean_text(" ".join(labs))
    labels_text.append(labs_text)

# Criar os tokenizadores com um token para palavras fora do vocabulário (<UNK>)
label_tokenizer = Tokenizer(oov_token="<UNK>")
label_tokenizer.fit_on_texts(labels_text)

# Para as descrições, usamos filters='' para não remover nenhum caractere e preservar os tokens especiais
description_tokenizer = Tokenizer(oov_token="<UNK>", filters='')
description_tokenizer.fit_on_texts(descriptions)

# Converter os textos em sequências numéricas
label_seq = label_tokenizer.texts_to_sequences(labels_text)
desc_seq = description_tokenizer.texts_to_sequences(descriptions)

# Definir comprimentos máximos:
# Limitamos o tamanho para evitar sequências muito longas (ex.: 20 para labels, 50 para descrições)
max_label_length = min(max(len(seq) for seq in label_seq), 20)
max_desc_length = min(max(len(seq) for seq in desc_seq), 50)

# Aplicar padding para padronizar o tamanho das sequências
label_padded = pad_sequences(label_seq, maxlen=max_label_length, padding='post')
desc_seq_padded = pad_sequences(desc_seq, maxlen=max_desc_length, padding='post')

# Dividir os dados em conjuntos de treino e teste (80% treino e 20% teste)
label_train, label_test, desc_train, desc_test = train_test_split(
    label_padded, desc_seq_padded, test_size=0.2, random_state=42
)

# Salvar os dados pré-processados (em formato .npy)
np.save('./models/label_train.npy', label_train)
np.save('./models/label_test.npy', label_test)
np.save('./models/desc_train.npy', desc_train)
np.save('./models/desc_test.npy', desc_test)

# Salvar os tokenizadores em arquivos .pkl para uso futuro
with open('./models/label_tokenizer.pkl', 'wb') as handle:
    pickle.dump(label_tokenizer, handle)
with open('./models/description_tokenizer.pkl', 'wb') as handle:
    pickle.dump(description_tokenizer, handle)

# Salvar os parâmetros de pré-processamento (comprimentos máximos)
preprocess_params = {'max_label_length': max_label_length, 'max_desc_length': max_desc_length}
with open('./models/preprocess_params.pkl', 'wb') as handle:
    pickle.dump(preprocess_params, handle)

print("Pré-processamento aprimorado concluído e dados salvos.")
#7.1s

GPU(s) detectada(s) e configurada(s).
Pré-processamento aprimorado concluído e dados salvos.


In [7]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge

# Carregar os tokenizadores
with open('./models/label_tokenizer.pkl', 'rb') as f:
    label_tokenizer = pickle.load(f)
with open('./models/description_tokenizer.pkl', 'rb') as f:
    description_tokenizer = pickle.load(f)

# Carregar os dados pré-processados
label_train = np.load('./models/label_train.npy')
label_test = np.load('./models/label_test.npy')
desc_train = np.load('./models/desc_train.npy')
desc_test = np.load('./models/desc_test.npy')

# Parâmetros do modelo
latent_dim = 512  # Dimensão do embedding definida para 512
label_vocab_size = len(label_tokenizer.word_index) + 1
desc_vocab_size = len(description_tokenizer.word_index) + 1

# Preparar os dados do decoder: entradas e targets (deslocados)
decoder_input_data = desc_train[:, :-1]
decoder_target_data = desc_train[:, 1:]
decoder_target_data = np.expand_dims(decoder_target_data, -1)

# Definir entradas do encoder e decoder
encoder_inputs = Input(shape=(None,), name='encoder_inputs')
decoder_inputs = Input(shape=(None,), name='decoder_inputs')

# Camadas de embedding
encoder_embedding_layer = Embedding(input_dim=label_vocab_size,
                                    output_dim=latent_dim,
                                    mask_zero=True,
                                    name='encoder_embedding')
decoder_embedding_layer = Embedding(input_dim=desc_vocab_size,
                                    output_dim=latent_dim,
                                    mask_zero=True,
                                    name='decoder_embedding')

# Encoder
encoder_embedding = encoder_embedding_layer(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True, name='encoder_lstm')
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_embedding = decoder_embedding_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name='decoder_lstm')
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(desc_vocab_size, activation='softmax', name='decoder_dense')
decoder_outputs = decoder_dense(decoder_outputs)

# Modelo Seq2Seq para treinamento
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Callbacks para treino
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=3)
]

# Treinar o modelo
model.fit(
    [label_train, decoder_input_data],
    decoder_target_data,
    batch_size=256,
    epochs=50,
    validation_split=0.2,
    callbacks=callbacks
)

# Salvar o modelo treinado
model.save('./models/seq2seq_model.keras')

# Construir o modelo do encoder para inferência
encoder_model = Model(encoder_inputs, encoder_states)

# Construir o modelo do decoder para inferência
decoder_state_input_h = Input(shape=(latent_dim,), name='input_h')
decoder_state_input_c = Input(shape=(latent_dim,), name='input_c')
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)
decoder_outputs_inf, h_inf, c_inf = decoder_lstm(decoder_embedding_inf, initial_state=decoder_states_inputs)
decoder_states_inf = [h_inf, c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

# Função de inferência
def decode_sequence(input_seq, max_length=50):
    states_value = encoder_model.predict(input_seq)
    if 'startseq' not in description_tokenizer.word_index:
        raise ValueError("Token 'startseq' não encontrado no vocabulário.")
    target_seq = np.array([[description_tokenizer.word_index['startseq']]])
    stop_condition = False
    decoded_sentence = []
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = description_tokenizer.index_word.get(sampled_token_index, '')
        if sampled_word == 'endseq' or len(decoded_sentence) >= max_length:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]
    return ' '.join(decoded_sentence)

# Exemplos de inferência
generated_descriptions = []
print("\nExemplos de inferência:")
for seq in label_test[:10]:
    input_seq = seq.reshape(1, -1)
    decoded_sentence = decode_sequence(input_seq)
    generated_descriptions.append(decoded_sentence)
    print("Gerado:", decoded_sentence)

# Funções de avaliação: BLEU e ROUGE
def calculate_bleu(reference, candidate):
    return sentence_bleu([reference.split()], candidate.split())

def calculate_rouge(reference, candidate):
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-1']['f']

# Calcular métricas para os primeiros exemplos
bleu_scores = []
rouge_scores = []
for i in range(min(len(desc_test), len(generated_descriptions))):
    reference_tokens = [description_tokenizer.index_word.get(idx, '') for idx in desc_test[i] if idx != 0]
    reference_sentence = ' '.join(reference_tokens)
    candidate_sentence = generated_descriptions[i]
    bleu_scores.append(calculate_bleu(reference_sentence, candidate_sentence))
    rouge_scores.append(calculate_rouge(reference_sentence, candidate_sentence))

print(f"\nBLEU Score Médio: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"ROUGE Score Médio: {sum(rouge_scores)/len(rouge_scores):.4f}")

# Resultado esperado
# The hypothesis contains 0 counts of 4-gram overlaps.
# Therefore the BLEU score evaluates to 0, independently of
# how many N-gram overlaps of lower order it contains.
# Consider using lower n-gram order or use SmoothingFunction()
#  warnings.warn(_msg)
# Tempo de execucao SEM GPU: 1h 15min (aproximadamente)
# Epoch 50/50
# BLEU Score Médio: 0.1953
# ROUGE Score Médio: 0.4369


ModuleNotFoundError: No module named 'rouge'